> **Credentials:** set `KAGGLE_USERNAME`, `KAGGLE_KEY`, and `WANDB_API_KEY` in the environment.
> Never hardcode credentials in this notebook.


# TR_ABDOMEN_RAD smoke test

Mini version of the staged notebook: 500 train / 200 val / 50 test images, 1 epoch per session, YOLO26-L-P2 @1280 on 2xT4.
Tests DDP, wandb logging, the watchdog, and the checkpoint chain (separate `huseyincavus/abdomen-ckpt-smoke` dataset).
Push twice: run 1 fresh, run 2 resumes.

In [ ]:
import os
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))
print("input:", sorted(os.listdir("/kaggle/input")))

In [ ]:
!pip -q install ultralytics==8.4.93 kagglehub kaggle wandb scikit-learn

In [ ]:
import json
import os
from pathlib import Path

import wandb

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
if not (KAGGLE_USERNAME and KAGGLE_KEY):
    raise SystemExit("Set KAGGLE_USERNAME and KAGGLE_KEY in the environment")

kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
kaggle_path.parent.mkdir(parents=True, exist_ok=True)
kaggle_path.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_path.chmod(0o600)

wandb_key = os.environ.get("WANDB_API_KEY")
wandb.login(key=wandb_key, relogin=True) if wandb_key else wandb.login()


In [ ]:
import os

def pick(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    return None

PREPARED = pick([
    "/kaggle/input/abdomen-prepared/yolo_dataset/images",
    "/kaggle/input/datasets/huseyincavus/abdomen-prepared/yolo_dataset/images",
])
if PREPARED is None:
    raise FileNotFoundError("abdomen-prepared images not mounted")

LABELS = pick([
    "/kaggle/input/abdomen-yolo-labels/labels",
    "/kaggle/input/datasets/huseyincavus/abdomen-yolo-labels/labels",
])
if LABELS is None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("huseyincavus/abdomen-yolo-labels", path="/kaggle/working/labels_dl", unzip=True, quiet=True)
    LABELS = "/kaggle/working/labels_dl/labels"
print("prepared:", PREPARED)
print("labels:", LABELS)

In [ ]:
import os
import shutil

code_root = None
for d in (
    "/kaggle/input/tr-abdomen-rad-code",
    "/kaggle/input/datasets/axqetc/tr-abdomen-rad-code",
    "/kaggle/input/datasets/huseyincavus/tr-abdomen-rad-code",
):
    if os.path.isdir(d):
        code_root = d
        break
if code_root is None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("axqetc/tr-abdomen-rad-code", path="/kaggle/working/code_dl", unzip=True, quiet=True)
    code_root = "/kaggle/working/code_dl"
print("code_root:", code_root)
for d in ("src", "configs", "trainer", "annotations"):
    shutil.copytree(os.path.join(code_root, d), f"/kaggle/working/{d}", dirs_exist_ok=True)
print(sorted(os.listdir("/kaggle/working")))

In [ ]:
from pathlib import Path

DS = Path("/kaggle/working/ds")
for split in ("train", "val", "test"):
    for kind, src in (("images", PREPARED), ("labels", LABELS)):
        dst = DS / kind / split
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            dst.symlink_to(f"{src}/{split}")
print("train imgs:", len(list((DS / "images" / "train").glob("*.jpg"))))

In [ ]:
import os
import subprocess
from pathlib import Path

from ultralytics import YOLO

subprocess.run(["python", "/kaggle/working/src/make_subset.py",
                "--src", "/kaggle/working/ds", "--out", "/kaggle/working/ds_mini",
                "--train", "500", "--val", "200", "--test", "50", "--per-patient", "2"], check=True)

TARGET_EPOCHS = 1
CKPT = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("axqetc/abdomen-ckpt-smoke", path="/kaggle/working/ckpt", unzip=True, quiet=True)
    for root, _, files in os.walk("/kaggle/working/ckpt"):
        if "last.pt" in files:
            CKPT = os.path.join(root, "last.pt")
            break
except Exception as e:
    print("no smoke checkpoint yet:", str(e)[:120])

def stop_after_epoch(trainer):
    trainer._session_epochs = getattr(trainer, "_session_epochs", 0) + 1
    if trainer._session_epochs >= TARGET_EPOCHS:
        trainer.stop = True

if CKPT:
    print("resuming from", CKPT, flush=True)
    model = YOLO(CKPT)
    model.add_callback("on_train_epoch_end", stop_after_epoch)
    model.train(resume=True)
else:
    model = YOLO("/kaggle/working/configs/yolo26l-p2.yaml")
    model.add_callback("on_train_epoch_end", stop_after_epoch)
    model.train(
        data="/kaggle/working/ds_mini/dataset.yaml",
        pretrained="yolo26l.pt",
        epochs=3, imgsz=1280, batch=8, device=[0, 1],
        optimizer="AdamW", lr0=0.0007, cos_lr=True,
        mosaic=1.0, close_mosaic=1, fliplr=0.0,
        save_period=1, project="tr-abdomen-rad-smoke", name="smoke",
        seed=0, workers=2, cache=False, patience=100, exist_ok=True,
    )
print("training done", flush=True)

In [ ]:
import json
import shutil
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

LAST = Path("/kaggle/working/tr-abdomen-rad-smoke/smoke/weights/last.pt")
stage = Path("/kaggle/working/ckpt_upload")
if LAST.exists():
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    shutil.copy2(LAST, stage / "last.pt")
    (stage / "dataset-metadata.json").write_text(json.dumps({
        "title": "abdomen_ckpt_smoke",
        "id": "axqetc/abdomen-ckpt-smoke",
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": True,
        "description": "Smoke-test checkpoint for TR_ABDOMEN_RAD staged training.",
    }, indent=2))
    api = KaggleApi()
    api.authenticate()
    try:
        r = api.dataset_create_version(str(stage), version_notes="smoke checkpoint", dir_mode="zip", quiet=True)
    except Exception:
        r = api.dataset_create_new(str(stage), public=False, quiet=True, dir_mode="zip")
    print("ckpt upload:", getattr(r, "status", None), flush=True)
else:
    print("no last.pt to upload", flush=True)

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/tr-abdomen-rad-smoke/smoke/results.csv")
if p.exists():
    print(p.read_text())
w = Path("/kaggle/working/tr-abdomen-rad-smoke/smoke/weights")
print(sorted(x.name for x in w.iterdir()) if w.exists() else "no weights")